In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import Row
from delta import *
from pyspark.sql.types import LongType, StringType, StructField, StructType, BooleanType, ArrayType, IntegerType, DoubleType

# Importar TODAS as funções que vamos precisar para a normalização
from pyspark.sql.functions import col, trim, when, lit, lower, translate, regexp_replace

# Define o utilizador HDFS
os.environ['HADOOP_USER_NAME'] = 'hadoop'

print("Imports concluídos.")

Imports concluídos.


In [2]:
# warehouse_location aponta para a pasta silver (como no molde)
warehouse_location = 'hdfs://hdfs-nn:9000//projeto/silver'

builder = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Python Spark - IMDB Unified Adaptations (Movie + TV_SHOW)") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport() \

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Session configurada.")

Spark Session configurada.


In [4]:
print("A ler a tabela 'projeto.imdb_movies_adaptations'")

# Ler a tabela Silver já normalizada
df_movies = spark.read.table("projeto.imdb_movie_adaptations")

print(f"Registos lidos de Filmes: {df_movies.count()}")
df_movies.show(5, truncate=False)

A ler a tabela 'projeto.imdb_movies_adaptations'
Registos lidos de Filmes: 204
+---------------------------------------+--------------+-----------------------------------------+------------------------------------+---------+--------------------------------------+----------------------+------+
|author                                 |authorLabel   |book                                     |bookLabel                           |imdbId   |movie                                 |movieLabel            |tmdbId|
+---------------------------------------+--------------+-----------------------------------------+------------------------------------+---------+--------------------------------------+----------------------+------+
|NA                                     |NA            |http://www.wikidata.org/entity/q917117   |l.a. takedown                       |tt0113277|http://www.wikidata.org/entity/q42198 |heat                  |949   |
|http://www.wikidata.org/entity/q171091 |philip k. dick|http:

In [5]:
print("A ler a tabela 'projeto.imdb_tvshow_adaptations'...")

# Ler a tabela Silver já normalizada
df_tvshows = spark.read.table("projeto.imdb_tvshow_adaptations")

print(f"Registos lidos de Séries: {df_tvshows.count()}")
df_tvshows.show(5, truncate=False)

A ler a tabela 'projeto.imdb_tvshow_adaptations'...
Registos lidos de Séries: 1237
+-----------------------------------------+------------------+----------------------------------------+-------------------------------------+----------+-----------------------------------------+--------------------------------+------+
|author                                   |authorLabel       |book                                    |bookLabel                            |imdbId    |movie                                    |movieLabel                      |tmdbId|
+-----------------------------------------+------------------+----------------------------------------+-------------------------------------+----------+-----------------------------------------+--------------------------------+------+
|http://www.wikidata.org/entity/q35610    |arthur conan doyle|http://www.wikidata.org/entity/q1192552 |the adventure of the engineer's thumb|tt0091942 |http://www.wikidata.org/entity/q1853193  |the twentieth cent

In [6]:
print("--- Schema Movies (com type) ---")
df_movies.printSchema()

print("\n--- Schema TV Shows (com type) ---")
df_tvshows.printSchema()

--- Schema Movies (com type) ---
root
 |-- author: string (nullable = true)
 |-- authorLabel: string (nullable = true)
 |-- book: string (nullable = true)
 |-- bookLabel: string (nullable = true)
 |-- imdbId: string (nullable = true)
 |-- movie: string (nullable = true)
 |-- movieLabel: string (nullable = true)
 |-- tmdbId: string (nullable = true)


--- Schema TV Shows (com type) ---
root
 |-- author: string (nullable = true)
 |-- authorLabel: string (nullable = true)
 |-- book: string (nullable = true)
 |-- bookLabel: string (nullable = true)
 |-- imdbId: string (nullable = true)
 |-- movie: string (nullable = true)
 |-- movieLabel: string (nullable = true)
 |-- tmdbId: string (nullable = true)



In [7]:
# Unir as tabelas baseando-se nos nomes das colunas
df_imdb_unified = df_movies.unionByName(df_tvshows)

print("União concluída.")

União concluída.


In [8]:
count_movies = df_movies.count()
count_tv = df_tvshows.count()
count_total = df_imdb_unified.count()

print(f"Registos Filmes:   {count_movies}")
print(f"Registos Séries:   {count_tv}")
print(f"Registos Totais:   {count_total}")

is_match = (count_movies + count_tv) == count_total
print(f"\nVerificação (Soma == Total?): {is_match}")

if not is_match:
    print("ALERTA: A soma das contagens não bate com o total!")

Registos Filmes:   204
Registos Séries:   1237
Registos Totais:   1441

Verificação (Soma == Total?): True


In [9]:
print("\nMostra de dados unificados:")
df_imdb_unified.show(10, truncate=False)


Mostra de dados unificados:
+---------------------------------------+-----------------------+-----------------------------------------+------------------------------------+---------+--------------------------------------+------------------------------+------+
|author                                 |authorLabel            |book                                     |bookLabel                           |imdbId   |movie                                 |movieLabel                    |tmdbId|
+---------------------------------------+-----------------------+-----------------------------------------+------------------------------------+---------+--------------------------------------+------------------------------+------+
|NA                                     |NA                     |http://www.wikidata.org/entity/q917117   |l.a. takedown                       |tt0113277|http://www.wikidata.org/entity/q42198 |heat                          |949   |
|http://www.wikidata.org/entity/q171091 |ph

In [13]:
print("A gravar a tabela unificada 'projeto.book_to_imdb'...")

df_imdb_unified.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/projeto.db/book_to_imdb") \
    .saveAsTable("projeto.book_to_imdb")

print("Tabela gravada com sucesso.")

A gravar a tabela unificada 'projeto.book_to_imdb'...
Tabela gravada com sucesso.


In [14]:
spark.sql(
    """
    DESCRIBE HISTORY projeto.book_to_imdb
    """
).show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2025-12-28 13:01:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|          0|  Serializable|        false|{numFiles -> 2, n...|        null|Apache-Spark/3.4....|
|      0|2025-12-28 13:00:...|  null|    null|CREATE OR REPLACE...|{isManaged -> fal...|null|    null|     null|       null|  Serializable|        false|{numFiles -

In [15]:
spark.sql(
    """
    SELECT * FROM projeto.book_to_imdb
    """
).show()

+--------------------+-------------------+--------------------+--------------------+----------+--------------------+--------------------+------+
|              author|        authorLabel|                book|           bookLabel|    imdbId|               movie|          movieLabel|tmdbId|
+--------------------+-------------------+--------------------+--------------------+----------+--------------------+--------------------+------+
|http://www.wikida...| arthur conan doyle|http://www.wikida...|the adventure of ...| tt0091942|http://www.wikida...|the twentieth cen...|261326|
|http://www.wikida...| melissa de la cruz|http://www.wikida...| witches of east end| tt2288064|http://www.wikida...| witches of east end| 60557|
|http://www.wikida...|     daisuke aizawa|http://www.wikida...|the eminence in s...|tt14115938|http://www.wikida...|the eminence in s...|119495|
|http://www.wikida...|        yoon tae-ho|http://www.wikida...|             misaeng| tt4240730|http://www.wikida...|             m

In [8]:
spark.stop()